# 09 · CP4 generalization gate, seed 42: P1 on the development split (P2 waits)

**L4.** Run it after notebook 07 has frozen the development split and notebook 08 has cached the unified probe.

- **P1 dev** (20% of each GenImage validation folder, frozen):
  - the detection model and the detection CLIP probe (G1, RF);
  - the unified model and the unified probe (second row);
  - raw and harmonized files. Real images are scored once, since they are the same in both versions.
- **G1:** detection model mean accuracy > 0.721, and above the probe on raw and on harmonized files.
- **RF:** ≥ 0.98 on every unseen generator → stop and ask.
- **P2:** behind `P2_ENABLED = False` until Dhanush decides on localization.
- **Also reported:** the unified model's false "edited" rate on real images by tile count; published GenImage results for context.

**After the CP4 fix:** set `RUN_EXPERIMENT = 'eviground_ctx'` (the unified model of notebook 10; detection stays CP3's) and `P2_ENABLED` as decided. Results then get the suffix `_ctx`.

**Result** (send to Claude): `Paper 1/runs/checkpoints/cp4_generalization_gate_seed42.json` (`…_ctx.json` after the fix). Per-image P1 rows are written to `runs/cp4/p1_dev_seed42.csv`; the accuracy figure (seaborn, PNG + PDF) to `runs/cp4/figures/`. The last cell draws it from the saved report, so it can be redrawn on a CPU runtime: `FIGURES_ONLY = True` in cell 2, then cells 1, 2 and the last two.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
DRY_DIR = 'eviground_cp4gate_dry'
P2_ENABLED = False   # True only after Dhanush's decision on localization
P1_ENABLED = True    # False: P2 only; G1/RF and the P1 rows come from the saved CP3 gate file
                     # (cp4_generalization_gate_seed42.json; the detection model is unchanged)

import csv, datetime, json, logging, os, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import numpy as np, torch
from eviground.config import load_config
from eviground.data import pack
from eviground.env import get_paths, session_info
from eviground.eval import runner

CU_PER_HOUR = 1.94         # read the live rate in Colab's resource panel and update
RUN_EXPERIMENT = None      # None: the CP3 models; 'eviground_ctx': the unified model of the CP4 fix
                           # (notebook 10). Detection stays CP3's. Result files get the suffix '_ctx'.
FIGURES_ONLY = False       # True: only redraw the figures from the saved files (CPU runtime is enough);
                           # then run cells 1, 2 and the last two cells (figures, release)
SEED = 42
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, tiny models, CPU
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / DRY_DIR
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}']
cfg = load_config(CONFIG, OVERRIDES)
if not DRY and not FIGURES_ONLY:
    assert torch.cuda.is_available(), 'needs a GPU runtime (L4); or set FIGURES_ONLY = True to redraw figures'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EXPERIMENT = cfg.get('experiment', 'eviground')
CK = paths.runs / 'checkpoints'
OUT = paths.runs / 'cp4'
PROBES = OUT / 'probes'
PROBE_EPOCHS = 1 if DRY else 3
WORKERS = 0 if DRY else int(cfg.data.num_workers)
BATCH = 8 if DRY else 64
def run_dir(profile):
    exp = RUN_EXPERIMENT if RUN_EXPERIMENT and profile == 'unified' else EXPERIMENT
    return paths.runs / f'{exp}_{profile}_seed{SEED}'
RUN_TAG = f"_{RUN_EXPERIMENT.rsplit('_', 1)[-1]}" if RUN_EXPERIMENT else ''   # '_ctx' for the fix
OUT.mkdir(parents=True, exist_ok=True); PROBES.mkdir(parents=True, exist_ok=True)
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| device', DEVICE,
      '| unified model', run_dir('unified'), '| result suffix', repr(RUN_TAG))
from eviground.eval import cp4
cp4.check_run_settings(P2_ENABLED, RUN_EXPERIMENT, DRY)   # P2 on the CP3 model already ran once by mistake
PROFILES = ['unified', 'detection'] if P1_ENABLED else ['unified']
print('P1', 'scored' if P1_ENABLED else 'reused from the saved CP3 gate file', '| P2', 'on' if P2_ENABLED else 'off',
      '| result file', f'cp4_generalization_gate_seed{SEED}{RUN_TAG}.json')

In [ ]:
# Cell 2b: pre-flight. Prints what runs and the estimated CU; stops before any staging if an input
# is missing or the estimate exceeds CU_LEFT (the balance shown in Colab's resource panel; None: not checked)
from eviground.utils import session
CU_LEFT = None
EST_CU = 0.0 if (DRY or FIGURES_ONLY) else (2.8 if P1_ENABLED else 0.3)
PREFLIGHT = session.preflight(
    {'notebook': '09 CP4 gate', 'unified model': run_dir('unified'), 'result file': f'cp4_generalization_gate_seed{SEED}{RUN_TAG}.json',
     'figures only': FIGURES_ONLY},
    paths={} if DRY else {**{f'{p} checkpoint': run_dir(p) / 'best.pt' for p in PROFILES}, 'dev split': CK / 'cp4_dev_split.json', **({} if P1_ENABLED else {'saved CP3 gate file': CK / f'cp4_generalization_gate_seed{SEED}.json'})}, est_cu=EST_CU, cu_left=CU_LEFT)

In [ ]:
# Cell 3: shards (dry run: fake world and tiny trained models)

assert not FIGURES_ONLY, 'FIGURES_ONLY = True: run only cells 1, 2 and the last two cells (figures, release)'
SHARDS = paths.shards
CP1_RESULT = None
if DRY:   # fake world -> CP1 -> shards -> tiny trained models (one epoch each)
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    from eviground.train.trainer import Trainer
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    CP1_RESULT = run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                                          base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
                         out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(CK / 'cp1' / 'manifest_cp1.csv.gz', SHARDS, free_bytes=10**12, margin=0,
              shard_bytes=200_000, sample_per_source=5, progress=None)
    for profile in PROFILES:
        Trainer(load_config(CONFIG, OVERRIDES + ['train.epochs=1']), SHARDS, run_dir(profile),
                seed=SEED, profile=profile, resume=None, log=None).fit()
LOCAL_SHARDS = paths.local_data / 'shards'
print(pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print))
SHARDS = LOCAL_SHARDS

In [ ]:
# Cell 4: the frozen development split, the manifest and the GenImage quality distribution
from eviground.data import harmonize
from eviground.data.cp1 import _read_csv_gz
from eviground.eval import devsplit, testsets
MANIFEST = _read_csv_gz(CK / 'cp1' / 'manifest_cp1.csv.gz')
if DRY:   # the fake world has 2 images per folder: a larger fraction so something is scored
    SPLIT = devsplit.build_dev_split(MANIFEST, fraction=0.5)
    table = CP1_RESULT['quality_distribution_genimage_real']['table']
else:
    SPLIT = devsplit.load(CK / 'cp4_dev_split.json')
    assert devsplit.verify(SPLIT, MANIFEST)['same'], 'the frozen dev split does not match the manifest'
    table = json.loads((CK / 'cp1_data_integrity.json').read_text())['result']['quality_distribution_genimage_real']['table']
DIST = harmonize.QualityDistribution.from_dict(table)
P1_ROWS, STAGE_MIN = [], 0.0
if P1_ENABLED:
    P1_ROWS = testsets.dev_rows(MANIFEST, SPLIT['dev_uids'], 'test_P1')
    t = time.time()
    P1_ROWS = testsets.stage_files(P1_ROWS, paths.local_data / 'p1_dev')
    STAGE_MIN = round((time.time() - t) / 60, 2)
print(len(P1_ROWS), 'P1 dev images staged in', STAGE_MIN, 'min | split', SPLIT['dev_sha256'][:12])

In [ ]:
# Cell 5: models and probes

from eviground.baselines import clip_probe

def get_probe(profile, model):
    '''The cached probe of ``profile``, or train it (the model's frozen CLIP, LoRA off).'''
    path = PROBES / f'probe_{profile}.pt'
    if path.is_file():
        saved = torch.load(path, weights_only=True)
        print('probe reused:', path)
        return clip_probe.ProbeHead.from_state(saved['head']).to(DEVICE), saved['report']
    tc = dict(cfg.train)
    crops = int((tc.get('detection') or {}).get('crops_per_epoch', 16000)) if profile == 'detection' else int(tc['crops_per_epoch'])
    train_b, val_b = runner.probe_data(SHARDS, profile=profile, crop=int(cfg.data.crop),
                                       crops_per_epoch=crops, epochs=PROBE_EPOCHS, seed=SEED,
                                       batch=BATCH, workers=WORKERS, augment_cfg=cfg.data.get('augment'))
    t = time.time()
    head, report = clip_probe.train_probes(model, train_b(), val_b, profile=profile, device=DEVICE)
    report['minutes'] = round((time.time() - t) / 60, 2)
    torch.save({'head': head.state(), 'report': report}, path)
    return head.to(DEVICE), report

MODELS, HEADS, PROBE_REPORTS = {}, {}, {}
for profile in PROFILES:   # with P1_ENABLED = False only the unified model (and its probe) is needed for P2
    MODELS[profile] = runner.load_trained(cfg.model, run_dir(profile) / 'best.pt', DEVICE)
    HEADS[profile], PROBE_REPORTS[profile] = get_probe(profile, MODELS[profile])
print({p: r.get('class_token') for p, r in PROBE_REPORTS.items()})

In [ ]:
# Cell 6: P1 dev, raw and harmonized (real images once); with P1_ENABLED = False, the saved CP3 rows
RECORDS, STATS = [], {}
if not P1_ENABLED:
    P1_SOURCE_FILE = CK / f'cp4_generalization_gate_seed{SEED}.json'
    _saved = json.loads(P1_SOURCE_FILE.read_text())
    _table, _probes, _ = cp4.p1_from_report(_saved)
    # only the detection rows: the saved unified rows belong to the CP3 unified model, not this one
    TABLE = {k: v for k, v in _table.items() if k.startswith('detection')}
    PROBE_REPORTS['detection'] = (_probes or {}).get('detection')
    FALSE_EDITED = None   # a unified-model P1 measure; not rerun here
    P1_SOURCE = {'file': P1_SOURCE_FILE.name, 'utc': _saved.get('utc'),
                 'git_commit': (_saved.get('session') or {}).get('git_commit'), 'p1_rerun': False}
    print('P1 reused from', P1_SOURCE)
real_rows = [r for r in P1_ROWS if r['label'] == 'real']
gen_rows = [r for r in P1_ROWS if r['label'] == 'generated']
for profile in (PROFILES if P1_ENABLED else []):
    predict = runner.joint_predictor(MODELS[profile], HEADS[profile], DEVICE)
    names = {'model': profile, 'probe': f'{profile}_probe'}
    for version, rows in (('raw', P1_ROWS), ('harmonized', gen_rows)):
        recs, STATS[f'{profile}/{version}'] = runner.p1_pass(
            predict, testsets.file_items(rows, version=version, dist=DIST), DEVICE, names, version)
        RECORDS += recs
        print(profile, version, STATS[f'{profile}/{version}'])
# the real images are the same files in both versions
RECORDS += [{**r, 'version': 'harmonized'} for r in RECORDS if r['version'] == 'raw' and r['label'] == 'real']
if P1_ENABLED:
    TABLE, FALSE_EDITED = cp4.p1_table(RECORDS), cp4.false_edited(RECORDS, 'unified')
    P1_SOURCE = {'p1_rerun': True}
    with open(OUT / f'p1_dev_seed{SEED}{RUN_TAG}.csv', 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=list(RECORDS[0]))
        w.writeheader(); w.writerows(RECORDS)

In [ ]:
# Cell 7: P2 dev (only after Dhanush's decision)
P2 = None
if P2_ENABLED:
    from eviground.data.download import ensure_data
    if not DRY:
        ensure_data(datasets=['diffseg30k'], paths=paths, stage_local=True)
    p2_rows = testsets.dev_rows(MANIFEST, SPLIT['dev_uids'], 'test_P2')
    predict = runner.joint_predictor(MODELS['unified'], HEADS['unified'], DEVICE)
    P2_DIAG = runner.diagnosis_pass(predict, testsets.file_items(p2_rows), DEVICE, ceilings=False, sweep=False)
    P2 = {'model': runner.p2_summary(P2_DIAG['edited'], 'model'),
          'probe': runner.p2_summary(P2_DIAG['edited'], 'probe'), 'pass': P2_DIAG['stats']}
    print(json.dumps(P2['model']['all'], indent=1, default=str))
else:
    print('P2 not scored (P2_ENABLED = False)')

In [ ]:
# Cell 8: the CP4 evaluation and its result file
hours = (time.time() - T0) / 3600
REPORT = cp4.evaluate(TABLE, model='detection', probe='detection_probe',
                      p2=None if P2 is None else P2['model']['all'],
                      extra={
    'step': 'CP4 generalization gate (development split)', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
    'session': info, 'dev_split': devsplit.summary(SPLIT), 'probes': PROBE_REPORTS,
    'false_edited_unified_p1_real': FALSE_EDITED, 'p1_source': P1_SOURCE,
    'p2': P2, 'stats': STATS, 'staging_minutes': STAGE_MIN,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR, 'compute_units': round(hours * CU_PER_HOUR, 2),
})
out = cp4.write_report(REPORT, CK / f'cp4_generalization_gate_seed{SEED}{RUN_TAG}.json')
for name, c in REPORT['checks'].items():
    print(f'{name:3} {c["status"]:13}', {k: v for k, v in c.items() if k != 'status'})
for head, versions in TABLE.items():
    print(head, {v: round(t['mean_acc'], 4) for v, t in versions.items()})
print('script verdict:', REPORT['verdict'], '| CU', REPORT['compute_units'], '| saved', out)

In [ ]:
# Cell 9: figure (seaborn), drawn from the saved report only
# To redraw it later without the GPU: a CPU runtime, FIGURES_ONLY = True in cell 2, then cells 1, 2, this one and the last.
from eviground import viz
saved_report = json.loads((CK / f'cp4_generalization_gate_seed{SEED}{RUN_TAG}.json').read_text())
FIG = viz.p1_figure(saved_report['p1'], OUT / 'figures', seed=f'{SEED}{RUN_TAG}')
print('figure (PNG + PDF):', FIG)
if DRY:
    assert FIG is not None and FIG.is_file(), FIG

In [ ]:
# Last cell: the session's compute record, then the runtime is released (all outputs are saved above)
COMPUTE = session.finish(OUT, T0, CU_PER_HOUR, name=f'compute_cp4_gate_seed{SEED}{RUN_TAG}.json',
                         extra={'git_commit': info['git_commit'], 'gpu': info.get('gpu_name'),
                                'est_cu': EST_CU, 'figures_only': FIGURES_ONLY})
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED